In [ ]:
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from google.colab import files # For Colab file upload

# 1. CONSTANTS AND HYPERPARAMETERS
IMG_WIDTH, IMG_HEIGHT = 75, 75 # InceptionV3 minimum is 75x75. CIFAR-10 is 32x32, so we upscale.
# Using a larger size like 150x150 or 224x224 might yield better results but take longer.
BATCH_SIZE = 64
EPOCHS = 10 # Keep low for a quick run, increase for better performance
NUM_CLASSES = 10
LEARNING_RATE = 0.001

# 2. LOAD AND PREPROCESS CIFAR-10 DATASET
print("Loading CIFAR-10 dataset...")
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

print(f"Original x_train shape: {x_train.shape}") # (50000, 32, 32, 3)

# Resize images and preprocess for InceptionV3
def preprocess_data(images, labels, target_size=(IMG_WIDTH, IMG_HEIGHT)):
    # Resize images
    images_resized = tf.image.resize(images, target_size).numpy() # Use numpy() for immediate execution
    # Preprocess input for InceptionV3 (scales pixels between -1 and 1)
    images_preprocessed = tf.keras.applications.inception_v3.preprocess_input(images_resized)
    # Convert labels to one-hot encoding
    labels_categorical = to_categorical(labels, NUM_CLASSES)
    return images_preprocessed, labels_categorical

print("Preprocessing training data...")
x_train_processed, y_train_processed = preprocess_data(x_train, y_train)
print("Preprocessing test data...")
x_test_processed, y_test_processed = preprocess_data(x_test, y_test)

print(f"Processed x_train shape: {x_train_processed.shape}")
print(f"Processed y_train shape: {y_train_processed.shape}")

# CIFAR-10 class names
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Optional: Data Augmentation for training set
# train_datagen = ImageDataGenerator(
#     rotation_range=20,
#     width_shift_range=0.2,
#     height_shift_range=0.2,
#     shear_range=0.2,
#     zoom_range=0.2,
#     horizontal_flip=True,
#     fill_mode='nearest'
# )
# Note: If using ImageDataGenerator, you'd typically use model.fit(train_datagen.flow(...))
# For simplicity here, we'll train on the already preprocessed (but not augmented) data.
# If you enable augmentation, ensure it happens *after* resizing but *before* InceptionV3 preprocessing,
# or ensure the augmentation function handles InceptionV3 preprocessing.

# 3. BUILD THE MODEL (TRANSFER LEARNING WITH INCEPTIONV3)
print("Building the model with InceptionV3 base...")

# Define the input shape
input_tensor = Input(shape=(IMG_WIDTH, IMG_HEIGHT, 3))

# Load InceptionV3 pre-trained on ImageNet, without the top classification layer
base_model = InceptionV3(weights='imagenet', include_top=False, input_tensor=input_tensor)

# Freeze the layers of the base model
for layer in base_model.layers:
    layer.trainable = False

# Add custom classification head
x = base_model.output
x = GlobalAveragePooling2D()(x) # Converts features to a single vector per image
x = Dense(1024, activation='relu')(x) # Add a fully-connected layer
x = Dropout(0.5)(x) # Add dropout for regularization
predictions = Dense(NUM_CLASSES, activation='softmax')(x) # Final classification layer

# This is the model we will train
model = Model(inputs=base_model.input, outputs=predictions)

# 4. COMPILE THE MODEL
print("Compiling the model...")
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

# 5. TRAIN THE MODEL
print("Starting model training...")
history = model.fit(
    x_train_processed, y_train_processed,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.2 # Use part of training data as validation
    # Or provide validation_data=(x_val_processed, y_val_processed) if you create a separate validation set
)

# 6. EVALUATE THE MODEL ON THE TEST SET
print("\nEvaluating model on the test set...")
test_loss, test_accuracy = model.evaluate(x_test_processed, y_test_processed, verbose=1)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy*100:.2f}%")

# 7. PLOT TRAINING HISTORY
print("\nPlotting training history...")

plt.figure(figsize=(12, 4))

# Plot training & validation accuracy values
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

# Plot training & validation loss values
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model Loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')

plt.tight_layout()
plt.show()

# (Optional) Save the model
# model.save('googlenet_cifar10_transfer.h5')
# print("Model saved as googlenet_cifar10_transfer.h5")

# 8. TEST WITH A SINGLE UPLOADED IMAGE
def predict_single_image(model, class_names_list, target_size=(IMG_WIDTH, IMG_HEIGHT)):
    print("\nPlease upload an image for prediction:")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return

    file_path = list(uploaded.keys())[0]

    try:
        img = Image.open(file_path).convert('RGB') # Ensure image is RGB

        # Display the image
        plt.imshow(img)
        plt.axis('off')
        plt.title("Uploaded Image")
        plt.show()

        # Preprocess the image
        img_resized = img.resize(target_size)
        img_array = np.array(img_resized) # Convert PIL image to numpy array
        img_array_expanded = np.expand_dims(img_array, axis=0) # Add batch dimension
        img_preprocessed = tf.keras.applications.inception_v3.preprocess_input(img_array_expanded)

        # Make prediction
        predictions_array = model.predict(img_preprocessed)
        predicted_class_index = np.argmax(predictions_array[0])
        predicted_class_name = class_names_list[predicted_class_index]
        confidence = np.max(predictions_array[0]) * 100

        print(f"\nPredicted Class: {predicted_class_name}")
        print(f"Confidence: {confidence:.2f}%")

        print("\nTop Probabilities:")
        # Get top 3 predictions
        top_indices = np.argsort(predictions_array[0])[-3:][::-1]
        for i in top_indices:
            print(f"- {class_names_list[i]}: {predictions_array[0][i]*100:.2f}%")

    except Exception as e:
        print(f"Error processing image: {e}")

# Run single image prediction
predict_single_image(model, class_names)

print("\n--- End of Script ---")

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Rescaling, Dropout
from tensorflow.keras.preprocessing import image_dataset_from_directory
import matplotlib.pyplot as plt
import numpy as np
import os
import zipfile # To handle zip files

# --- 1. Setup and Constants ---
IMG_WIDTH = 128  # Target width for resizing images
IMG_HEIGHT = 128 # Target height for resizing images
IMAGE_SIZE = (IMG_WIDTH, IMG_HEIGHT)
BATCH_SIZE = 32
EPOCHS = 10      # Number of training epochs (can be increased for better accuracy)
DATASET_ZIP_NAME = 'my_dataset.zip' # The name of the zip file you will upload
DATASET_EXTRACT_PATH = 'dataset_content' # Folder where dataset will be extracted

# --- 2. Upload and Unzip Your Dataset ---
# (This part is for Google Colab)
from google.colab import files

print(f"Please upload your dataset ZIP file named '{DATASET_ZIP_NAME}'")
uploaded = files.upload()

if DATASET_ZIP_NAME in uploaded:
    print(f"'{DATASET_ZIP_NAME}' uploaded successfully!")
    # Unzip the file
    with zipfile.ZipFile(DATASET_ZIP_NAME, 'r') as zip_ref:
        zip_ref.extractall(DATASET_EXTRACT_PATH)
    print(f"Dataset extracted to '{DATASET_EXTRACT_PATH}'")

    # IMPORTANT: Identify the actual root directory of your classes
    # e.g., if your zip extracts to 'dataset_content/my_dataset_folder/class_a',
    # then dataset_dir should be 'dataset_content/my_dataset_folder'
    # We will try to find the first directory inside DATASET_EXTRACT_PATH that contains subdirectories (classes)

    extracted_folders = [f for f in os.listdir(DATASET_EXTRACT_PATH) if os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, f))]
    if not extracted_folders:
        print(f"No subdirectories found in {DATASET_EXTRACT_PATH}. Please ensure your ZIP file has a root folder containing class folders.")
        # Handle error or exit
        exit()

    # Assuming the first folder found within the extract path is the root of your dataset
    # Or, if the class folders are directly under DATASET_EXTRACT_PATH
    dataset_base_dir = os.path.join(DATASET_EXTRACT_PATH, extracted_folders[0])
    # Check if this base_dir itself contains class folders, or if class folders are directly under DATASET_EXTRACT_PATH
    potential_class_folders_in_base = [d for d in os.listdir(dataset_base_dir) if os.path.isdir(os.path.join(dataset_base_dir, d))]

    if not potential_class_folders_in_base: # Maybe classes are directly under DATASET_EXTRACT_PATH
        # Check if DATASET_EXTRACT_PATH has multiple subdirectories (potential class folders)
        direct_class_folders = [d for d in os.listdir(DATASET_EXTRACT_PATH) if os.path.isdir(os.path.join(DATASET_EXTRACT_PATH, d))]
        if len(direct_class_folders) > 1: # Heuristic: if multiple folders, assume they are classes
             dataset_dir = DATASET_EXTRACT_PATH
        else: # Otherwise, assume the first extracted folder is the one containing class folders
            dataset_dir = os.path.join(DATASET_EXTRACT_PATH, extracted_folders[0])
    else:
        dataset_dir = dataset_base_dir

    print(f"Using dataset directory: {dataset_dir}")

    if not os.path.exists(dataset_dir) or not os.listdir(dataset_dir):
        print(f"ERROR: Dataset directory '{dataset_dir}' is empty or does not exist. "
              "Please check your ZIP file structure. "
              "It should contain a root folder, and inside it, subfolders for each class.")
        exit() # Exit if dataset directory is not valid
else:
    print(f"ERROR: '{DATASET_ZIP_NAME}' not found in uploaded files. Please upload the correct file.")
    exit() # Exit if zip is not uploaded

# --- 3. Load and Preprocess Data (with resizing) ---
print("\nLoading and preprocessing data...")

# Create training dataset (80% of data)
# image_dataset_from_directory automatically infers class names from folder names
# and resizes images.
train_dataset = image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,  # 20% of data for validation
    subset="training",
    seed=123,              # Seed for shuffling and splitting
    image_size=IMAGE_SIZE, # Resize images to this size
    batch_size=BATCH_SIZE
)

# Create validation dataset (20% of data)
validation_dataset = image_dataset_from_directory(
    dataset_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)

# Get class names (these are inferred from the subfolder names)
class_names = train_dataset.class_names
num_classes = len(class_names)
print(f"Found classes: {class_names}")
print(f"Number of classes: {num_classes}")

# Configure dataset for performance
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_dataset.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
validation_dataset = validation_dataset.cache().prefetch(buffer_size=AUTOTUNE)

# --- 4. Build a Simple CNN Model ---
print("\nBuilding the model...")
model = Sequential([
    # Rescale pixel values from [0, 255] to [0, 1]
    Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)),

    # First Convolutional Block
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Third Convolutional Block (optional, can add more)
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),

    # Flatten the results to feed into a DNN
    Flatten(),

    # Dense layer
    Dense(128, activation='relu'),
    Dropout(0.5), # Dropout for regularization

    # Output layer
    Dense(num_classes, activation='softmax') # Use 'softmax' for multi-class classification
])

# --- 5. Compile the Model ---
print("\nCompiling the model...")
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy', # Use sparse_categorical_crossentropy if labels are integers
    metrics=['accuracy']
)

model.summary() # Print model structure

# --- 6. Train the Model ---
print("\nStarting model training...")
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=EPOCHS
)

# --- 7. Evaluate the Model (Optional, as validation accuracy is already tracked) ---
print("\nEvaluating model...")
loss, accuracy = model.evaluate(validation_dataset) # Or use a separate test set if available
print(f"Validation Loss: {loss:.4f}")
print(f"Validation Accuracy: {accuracy*100:.2f}%")

# --- 8. Plot Training History ---
print("\nPlotting training history...")
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']
epochs_range = range(EPOCHS)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.title('Training and Validation Accuracy')

plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.title('Training and Validation Loss')
plt.show()

# --- 9. Predict on a New Image ---
def predict_new_image(model_to_predict, class_names_list):
    print("\nUpload an image for prediction:")
    uploaded_image = files.upload()

    if not uploaded_image:
        print("No file uploaded.")
        return

    file_path = list(uploaded_image.keys())[0]

    try:
        # Load and preprocess the image
        img = tf.keras.preprocessing.image.load_img(
            file_path, target_size=IMAGE_SIZE # Resize to the model's expected input size
        )
        img_array = tf.keras.preprocessing.image.img_to_array(img)
        img_array = tf.expand_dims(img_array, 0) # Create a batch

        # Make prediction
        predictions = model_to_predict.predict(img_array)
        score = tf.nn.softmax(predictions[0]) # Apply softmax to get probabilities

        predicted_class = class_names_list[np.argmax(score)]
        confidence = 100 * np.max(score)

        plt.imshow(img)
        plt.title(f"Predicted: {predicted_class} ({confidence:.2f}% confidence)")
        plt.axis("off")
        plt.show()

        print(f"This image most likely belongs to '{predicted_class}' with a {confidence:.2f}% confidence.")

    except Exception as e:
        print(f"Error processing or predicting image: {e}")

# Make a prediction on a new image
predict_new_image(model, class_names)

print("\n--- End of Simple Classification Script ---")